# Model Training
- Load pre-cleaned data
- Data Preprocessing
- model building
- HyperParameter tuning
- Model training
- Model compraision
- Save best model

In [1]:
# imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn import metrics

In [2]:
# Load pre-cleaned data
data = pd.read_csv(r"D:\ML_Project\Customer_churn_predection\Data\cleaned_data.csv")
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,NaN,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,NaN,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,NaN,53.85,NaN,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,NaN,42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,NaN,70.70,151.65,Yes


In [3]:
# data-backup
df = data.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7053 entries, 0 to 7052
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7053 non-null   str    
 1   SeniorCitizen     7053 non-null   int64  
 2   Partner           7053 non-null   str    
 3   Dependents        7053 non-null   str    
 4   tenure            7053 non-null   int64  
 5   PhoneService      7053 non-null   str    
 6   MultipleLines     7048 non-null   str    
 7   InternetService   7050 non-null   str    
 8   OnlineSecurity    7053 non-null   str    
 9   OnlineBackup      7053 non-null   str    
 10  DeviceProtection  7053 non-null   str    
 11  TechSupport       7053 non-null   str    
 12  StreamingTV       7053 non-null   str    
 13  StreamingMovies   7053 non-null   str    
 14  Contract          7053 non-null   str    
 15  PaperlessBilling  7053 non-null   str    
 16  PaymentMethod     7050 non-null   str    
 17  Monthl

In [4]:
# remove null value from target
df.dropna(subset = ['Churn'], inplace = True)
df.isnull().sum()

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        5
InternetService      3
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        3
MonthlyCharges       4
TotalCharges        14
Churn                0
dtype: int64

In [5]:
# Feature, Target, Numeric and categorical Features
X = df.drop(columns = "Churn")
y = df["Churn"].map({"Yes":1, "No":0})
print(y.dtype)

# Numeric and categorica features
X["SeniorCitizen"] = X["SeniorCitizen"].map({1: "Yes", 0:"No"})
num_features = X.select_dtypes(include = "number").columns.to_list()
cat_features = X.select_dtypes(include = "object").columns.to_list()
print("Numeric Features: \n", num_features)
print("\nCategorical Features: \n", cat_features)

int64
Numeric Features: 
 ['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features: 
 ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12356\3094422041.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include = "object").columns.to_list()


## Data preprocessing

In [6]:
# Numeric Pipeline
num_pipeline = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown = "ignore", sparse_output = False))
])

# Combine pipeline
preprocessor = ColumnTransformer(transformers = [
    ("numeric", num_pipeline, num_features),
    ("categorical", cat_pipeline, cat_features)
])


In [7]:
# Split and preprocess data
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [8]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter = 1000, random_state = 42),
    "Random Forest": RandomForestClassifier(random_state = 42),
    "SVM": SVC(probability = True, random_state = 42),
    "AdaBoost": AdaBoostClassifier(random_state = 42),
    "XGBoost": XGBClassifier(random_state = 42, eval_metric = "logloss")
}

In [9]:
# Define Hyperparameters
param_grid = {
    "Logistic Regression": {
        "model__C":[0.1,1,10]
    },
    "Random Forest": {
        "model__n_estimators":[50,100,200],
        "model__max_depth":[None, 5, 10],
        "model__min_samples_leaf":[1,2,5]
    },
    "SVM": {
        "model__C": [0.1,1,10],
        "model__kernel":["linear", "rbf"]
    },
    "AdaBoost": {
        "model__n_estimators":[50,100,200],
        "model__learning_rate":[0.05,0.5,1]
    },
    "XGBoost":{
        "model__n_estimators":[50,100,200],
        "model__max_depth":[3,5,10],
        "model__learning_rate":[0.05,0.5,1]
    }
}

In [10]:
# Model Training & Hyperparameter tuning
results = []
best_models = {}
for model_name, model in models.items():
    print(f"Model Training: {model_name}")

    # Complete pipeline
    pipe = Pipeline(steps = [
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Hyperparameter Tuning
    grid = GridSearchCV(estimator = pipe, param_grid = param_grid[model_name], scoring ="f1",
                       cv = 5, n_jobs = -1)
    # Train
    grid.fit(X_train, y_train)

    # Store best pipeline
    best_models[model_name] = grid.best_estimator_

    # Prediction
    y_pred = grid.predict(X_test)

    # Predict Probality
    y_prob = grid.predict_proba(X_test)[:,1]

    # Store result
    results.append({
        "Model":model_name,
        "Accuracy":metrics.accuracy_score(y_test, y_pred),
        "Precision": metrics.precision_score(y_test, y_pred),
        "Recall": metrics.recall_score(y_test, y_pred),
        "F1 Score": metrics.f1_score(y_test, y_pred),
        "ROC-AUC": metrics.roc_auc_score(y_test, y_prob),
        "Best Parameters": grid.best_params_
    })


Model Training: Logistic Regression
Model Training: Random Forest
Model Training: SVM


D:\ML_Project\Customer_churn_predection\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Model Training: AdaBoost
Model Training: XGBoost


In [12]:
import joblib

In [14]:
# Model Comparison
result_df = pd.DataFrame(results)

# Sort the result dataframe by F1 Score (best model first)
result_df = result_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

print("Model Comparison:\n")
print(result_df)

# Select the best model from the sorted result dataframe
best_model_name = result_df.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("\nBest Model:", best_model_name)
print("Best Parameters:", result_df.iloc[0]["Best Parameters"])

# Save the complete preprocessing + model pipeline
TRAINED_MODEL_PATH = r"D:\ML_Project\Customer_churn_predection\best_model.pkl"

joblib.dump(best_model, TRAINED_MODEL_PATH)

print(f"\nBest model saved successfully to: {TRAINED_MODEL_PATH}")


Model Comparison:

                 Model  Accuracy  Precision    Recall  F1 Score   ROC-AUC  \
0        Random Forest  0.792199   0.618729  0.508242  0.558069  0.824408   
1  Logistic Regression  0.786525   0.599369  0.521978  0.558003  0.818055   
2                  SVM  0.784397   0.594937  0.516484  0.552941  0.809858   
3             AdaBoost  0.785816   0.603333  0.497253  0.545181  0.825067   
4              XGBoost  0.782979   0.594771  0.500000  0.543284  0.825384   

                                     Best Parameters  
0  {'model__max_depth': 10, 'model__min_samples_l...  
1                                   {'model__C': 10}  
2        {'model__C': 10, 'model__kernel': 'linear'}  
3  {'model__learning_rate': 1, 'model__n_estimato...  
4  {'model__learning_rate': 0.05, 'model__max_dep...  

Best Model: Random Forest
Best Parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 5, 'model__n_estimators': 100}

Best model saved successfully to: D:\ML_Project\Customer_chu